# Early-Detection GELSTM (First-N Visits) — CLASSIFIER

**True early-detection setup.** For every subject, keep only the **first N earliest** visits (chronologically), drop subjects with fewer than N visits, and train/evaluate the LSTM on that fixed window. The subject's converter status is fixed at recruitment — only the *features the model sees* are restricted.

Sweeps `N ∈ {2, 3}` and compares against the **all-visits trajectory baseline** from the production notebook.

**Reading the table.**
* AUC stays high at N=2 → the model genuinely identifies converters from early scans (publishable early-detection result).
* AUC collapses at N=2 → prior results relied on later visits and/or sequence-length cues; reframe as *trajectory classification*.

In [ ]:
# === Papermill parameters (injected by run_experiment.py) ===
# Safe interactive defaults: None keeps the original Jupyter behaviour
# (interactive checkpoint/threshold prompts, JSON-config loading).
EXPERIMENT_ID = None
MODE = None
MODEL = None
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # None -> interactive checkpoint picker
THRESHOLD_MODE = None         # None -> interactive prompt; else youden | best-f1 | fixed
FIXED_THRESHOLD = None        # required when THRESHOLD_MODE is fixed
WANDB_ENABLED = True          # W&B logging is on by default
OUTPUT_DIR = None             # defaults to outputs/<experiment-id>/ when run standalone
RESOLVED_CONFIG = None        # merged hyperparameter dict; overrides on-disk JSON when set
RUN_DIR = None                # set by the runner: where run_summary.json / artifacts go
RUN_NAME = None               # set by the runner: the W&B run name


In [ ]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))

In [ ]:
# v2 reproducibility seeding — must run before datasets, samplers, or models.
from CLASSIFIER.common.seeding import (
    set_seed, make_rng, make_torch_generator, seed_worker,
)
SEED = 42
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)


In [ ]:
import json, copy, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

V2_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from model.GELSTM.models  import GELSTMClassifier
from model.GELSTM.dataset import LongitudinalSubjectDataset
from model.GELSTM.train   import train_epoch, evaluate, make_batches
from model.GELSTM.utils   import compute_class_weights
from common.sanity        import run_full_audit

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import sys
if '/mnt/e/fyassine/ad-early-detection' not in sys.path:
    sys.path.insert(0, '/mnt/e/fyassine/ad-early-detection')
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths

DATA_VERSION = '__fc_wholebrain_sch200_flat__'
DATA_ROOT    = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE') / DATA_VERSION
WB_DATA_ROOT = str(DATA_ROOT / 'matrices')
METADATA_DIR = DATA_ROOT / 'metadata'
COHORTS_CSV  = str(METADATA_DIR / 'cohorts.csv')
SPLITS_DIR   = splits_dir('downstream')
SPLIT_CSVS = {
    'train': str(SPLITS_DIR / 'train.csv'),
    'val':   str(SPLITS_DIR / 'val.csv'),
    'test':  str(SPLITS_DIR / 'test.csv'),
}
_ = run_full_audit(SPLIT_CSVS)

GAAE_CONFIG_PATH = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/configs/gaae_delcode_whole_brain.json')
GAAE_CKPT_PATH   = GAAE_CHECKPOINT_PATH or ''   # runner-supplied GAAE encoder, else none
with open(GAAE_CONFIG_PATH) as f:
    hp = json.load(f)
ADJ_K        = hp.get('adjacency_k', 8)
FILE_VARIANT = hp.get('file_variant', 'z_transformed')
HIDDEN       = hp.get('hidden_dim', 128)
LATENT       = hp.get('latent_dim', 64)
HEADS        = hp.get('num_heads', 2)
COND_DIM     = hp.get('cond_dim', 2)
DROPOUT      = hp.get('dropout', 0.3)

# Training hyper-params (same as production notebook).
BATCH_SIZE = 16
EPOCHS     = 50
PATIENCE   = 15
LR         = 1e-3
N_FOLDS    = 5

In [ ]:
# v2 split-hygiene audit — hard-fails if any subject crosses splits.
import sys
from pathlib import Path
_REPO_ROOT = Path('/mnt/e/fyassine/ad-early-detection')
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
_V2_ROOT = _REPO_ROOT / 'CLASSIFIER'
if str(_V2_ROOT) not in sys.path:
    sys.path.insert(0, str(_V2_ROOT))
from common.sanity import run_full_audit
from DATA.src.splitting.load_splits import split_csv_paths
_ = run_full_audit(split_csv_paths('downstream'))


In [ ]:
train_df = pd.read_csv(SPLIT_CSVS['train'])
val_df   = pd.read_csv(SPLIT_CSVS['val'])
test_df  = pd.read_csv(SPLIT_CSVS['test'])

cv_pool_df = pd.concat([train_df, val_df], ignore_index=True)
test_mci   = test_df.copy()

def build_datasets(max_visits, require_full_window):
    cv  = LongitudinalSubjectDataset(
        WB_DATA_ROOT, cv_pool_df, COHORTS_CSV,
        adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
        max_visits=max_visits, require_full_window=require_full_window,
    )
    te = LongitudinalSubjectDataset(
        WB_DATA_ROOT, test_mci, COHORTS_CSV,
        adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
        max_visits=max_visits, require_full_window=require_full_window,
    )
    return cv, te

## Train + evaluate for each window

In [ ]:
def build_model():
    m = GELSTMClassifier(
        in_features=200, gaae_hidden=HIDDEN,
        gaae_latent=LATENT, gaae_heads=HEADS,
        gaae_cond_dim=COND_DIM, gaae_dropout=DROPOUT,
        lstm_hidden=128, lstm_layers=2, lstm_dropout=0.3,
        use_time_delta=True, classifier_hidden=64,
    ).to(device)
    if GAAE_CKPT_PATH:
        m.load_gaae_weights(GAAE_CKPT_PATH, device=device)
    return m

from common import tracking
_wb_exp = {'id': EXPERIMENT_ID or 'gelstm-early-detection-first-n', 'mode': MODE or 'longitudinal',
           'model': MODEL or 'GELSTM', 'dataset': DATASET, 'seed': SEED, 'wandb': WANDB_ENABLED}
wandb_run = tracking.init_run(_wb_exp, {**(RESOLVED_CONFIG or {})})

def run_cv(items, labels, sids, n_folds=N_FOLDS):
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    fold_aucs = []
    oof_probs, oof_targets = [], []
    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(items, labels, groups=sids)):
        tr_items = [items[i] for i in tr_idx]
        va_items = [items[i] for i in va_idx]
        tr_labels = [labels[i] for i in tr_idx]
        pos_w  = compute_class_weights(tr_labels, device=device)
        crit   = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        m      = build_model()
        opt    = torch.optim.Adam(m.get_trainable_params(), lr=LR)
        best_auc, best_state, no_improve = 0.0, None, 0
        for epoch in range(EPOCHS):
            tr_b = make_batches(tr_items, BATCH_SIZE, shuffle=True)
            va_b = make_batches(va_items, BATCH_SIZE, shuffle=False)
            _    = train_epoch(m, tr_b, opt, crit, device)
            r    = evaluate(m, va_b, device)
            tracking.log_metrics(wandb_run, {'fold': fold+1, 'epoch': epoch+1, 'val_auc': r['auc']})
            if r['auc'] > best_auc:
                best_auc, best_state, no_improve = r['auc'], copy.deepcopy(m.state_dict()), 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break
        m.load_state_dict(best_state)
        va_b = make_batches(va_items, BATCH_SIZE, shuffle=False)
        r    = evaluate(m, va_b, device)
        fold_aucs.append(r['auc'])
        oof_probs.extend(r['probs'].tolist())
        oof_targets.extend(r['targets'].tolist())
        print(f'  Fold {fold+1}: AUC={r["auc"]:.4f}')
    return np.array(fold_aucs), np.array(oof_probs), np.array(oof_targets), m

rows = []
for max_v in [None, 3, 2]:
    label = 'all' if max_v is None else f'first_{max_v}'
    require_full = max_v is not None
    print(f'\n=== Window: {label}  (require_full_window={require_full}) ===')
    cv_ds, te_ds = build_datasets(max_visits=max_v, require_full_window=require_full)
    cv_items  = [cv_ds[i] for i in range(len(cv_ds))]
    cv_labels = cv_ds.get_labels()
    cv_sids   = cv_ds.get_subject_ids()
    te_items  = [te_ds[i] for i in range(len(te_ds))]

    fold_aucs, oof_p, oof_t, last_model = run_cv(cv_items, cv_labels, cv_sids)
    te_batches = make_batches(te_items, BATCH_SIZE, shuffle=False)
    te_metrics = evaluate(last_model, te_batches, device)

    rows.append({
        'window':       label,
        'cv_n':         len(cv_items),
        'test_n':       len(te_items),
        'cv_auc_mean':  float(fold_aucs.mean()),
        'cv_auc_std':   float(fold_aucs.std()),
        'test_auc':     float(te_metrics['auc']),
    })

summary = pd.DataFrame(rows)
summary

try:
    for _r in rows:
        tracking.log_metrics(wandb_run, {f"{_r['window']}_cv_auc": _r['cv_auc_mean'], f"{_r['window']}_test_auc": _r['test_auc']})
    if RUN_DIR:
        from common.provenance import write_run_summary, capture_git_provenance, capture_env
        write_run_summary(RUN_DIR, {'experiment_id': EXPERIMENT_ID, 'timestamp': RUN_NAME,
            'git': capture_git_provenance(), 'env': capture_env(),
            'metrics': {r['window'] + '_test_auc': r['test_auc'] for r in rows}})
    tracking.finish_run(wandb_run)
except NameError:
    pass


## Subject drop-out at each window

Document how many subjects are excluded by `require_full_window=True` for each N. A heavy drop-out at N=3 (e.g. > 30 % loss) limits the statistical strength of the first-N number even if the AUC is good.

In [ ]:
for max_v in [2, 3]:
    ds_open = LongitudinalSubjectDataset(
        WB_DATA_ROOT, cv_pool_df, COHORTS_CSV,
        adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
    )
    ds_strict = LongitudinalSubjectDataset(
        WB_DATA_ROOT, cv_pool_df, COHORTS_CSV,
        adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
        max_visits=max_v, require_full_window=True,
    )
    drop = len(ds_open) - len(ds_strict)
    print(f'N={max_v}: CV pool retains {len(ds_strict)}/{len(ds_open)} '
          f'({drop} dropped, {100*drop/max(len(ds_open),1):.1f}%)')